# Prompting Results — All Runs

Scans every saved prediction file, recomputes the metrics against the human labels, and prints
tables keyed by **model + provider + prompt + scope**. Nothing is re-run against an API; this only
reads files already on disk.

- Predictions: `outputs/prompting/predictions/`
- Combined results: `outputs/prompting/results/prompting_results_table.json`

Each run is scored on the scope it was run on. Runs still in progress are marked `(partial)`.

Because prompts are tuned on `dev` (train split only), a `dev` run shows no test coverage — the
held-out comparison comes from a separate `test` or `full` run of the frozen prompt.

In [ ]:
import json
import sys
from pathlib import Path
sys.path.insert(0, ".")

from src.common.config import PUBLISHED_BASELINES
from src.common.data import leakage_free_indices, load_bnsentmix, scope_indices, split_indices
from src.common.io_utils import load_jsonl
from src.common.metrics import sentiment_metrics
from src.prompting.prompts import parse_predictions_name

PREDICTIONS_DIR = Path("outputs") / "prompting" / "predictions"
OUTPUT_DIR = Path("outputs") / "prompting" / "results"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

dataset = load_bnsentmix()
index_map = split_indices(dataset)
SCORING_SETS = {
    "validation": sorted(index_map["validation"]),
    "test": sorted(index_map["test"]),
    "validation_clean": leakage_free_indices(index_map["validation"], index_map["train"], dataset),
    "test_clean": leakage_free_indices(index_map["test"], index_map["train"], dataset),
}

# every saved prediction file, whatever scope it was run on
runs = []
for path in sorted(PREDICTIONS_DIR.glob("*.jsonl")):
    provider, model, prompt_name, scope = parse_predictions_name(path)
    runs.append((provider, model, prompt_name, scope, path))

print(f"found {len(runs)} prediction files")
for provider, model, prompt_name, scope, path in runs:
    print(f"  {provider:<12}{model:<46}{prompt_name:<16}{scope}")


## Main table

In [ ]:
def load_run(path):
    records = {record["index"]: record for record in load_jsonl(path)}
    labelled = sum(1 for record in records.values() if record.get("prediction") is not None)
    return records, labelled


def score(records, indices):
    gold = [dataset[index]["Label"] for index in indices]
    predicted = [
        records[index]["prediction"]
        if index in records and records[index]["prediction"] is not None
        else -1
        for index in indices
    ]
    return sentiment_metrics(gold, predicted)


print("=" * 122)
print("PROMPTED SENTIMENT CLASSIFICATION - ALL RUNS (each scored on the scope it was run on)")
print("=" * 122)
header = (f"{'model':<38}{'provider':<13}{'prompt':<15}{'scope':<11}{'rows':>9}"
          f"{'accuracy':>10}{'precision':>11}{'recall':>9}{'f1':>9}")
print(header)
print("-" * len(header))

summary = {}
for provider, model, prompt_name, scope, path in runs:
    records, labelled = load_run(path)
    indices = scope_indices(scope, dataset)
    if labelled == 0:
        print(f"{model:<38}{provider:<13}{prompt_name:<15}{scope:<11}"
              f"{'0/' + str(len(indices)):>9}{'no labels yet':>39}")
        continue
    metrics = score(records, indices)
    weighted = metrics["weighted"]
    summary[(provider, model, prompt_name, scope)] = {
        "metrics": metrics,
        "labelled": labelled,
        "rows": len(indices),
        "path": path.name,
    }
    flag = "" if labelled >= len(indices) else "  (partial)"
    print(f"{model:<38}{provider:<13}{prompt_name:<15}{scope:<11}"
          f"{f'{labelled}/{len(indices)}':>9}"
          f"{weighted['accuracy']:>10.4f}{weighted['precision']:>11.4f}"
          f"{weighted['recall']:>9.4f}{weighted['f1']:>9.4f}{flag}")

print()
print("PUBLISHED BASELINES (reference paper, its own test split, weighted)")
print("-" * len(header))
for name in ("bert", "mbert", "banglabert", "banglishbert", "svm", "logistic-regression"):
    published = PUBLISHED_BASELINES[name]
    print(f"{published['paper_name']:<38}{'paper':<13}{'-':<15}{'test':<11}{'':>9}"
          f"{published['test']['accuracy']:>10.4f}{published['test']['precision']:>11.4f}"
          f"{published['test']['recall']:>9.4f}{published['test']['f1']:>9.4f}   (test)")


## Split breakdown

In [ ]:
print("=" * 110)
print("SPLIT COVERAGE - a dev run has no test coverage by construction, which is the point")
print("=" * 110)
header = (f"{'model':<38}{'prompt':<15}{'scope':<11}"
          f"{'validation':>12}{'test':>10}{'val_clean':>12}{'test_clean':>12}")
print(header)
print("-" * len(header))
for provider, model, prompt_name, scope, path in runs:
    records, labelled = load_run(path)
    if labelled == 0:
        continue
    line = f"{model:<38}{prompt_name:<15}{scope:<11}"
    for set_name in ("validation", "test", "validation_clean", "test_clean"):
        indices = [
            index
            for index in SCORING_SETS[set_name]
            if index in records and records[index]["prediction"] is not None
        ]
        width = 10 if set_name == "test" else 12
        if not indices:
            line += f"{'-':>{width}}"
            continue
        accuracy = score(records, indices)["weighted"]["accuracy"]
        line += f"{accuracy:>{width}.4f}"
    print(line)


## Per-class F1

In [ ]:
print("=" * 100)
print("PER-CLASS F1 (each run on its own scope) - Mixed is the hard class")
print("=" * 100)
header = (f"{'model':<38}{'prompt':<15}{'scope':<11}"
          f"{'Positive':>10}{'Negative':>10}{'Neutral':>9}{'Mixed':>8}")
print(header)
print("-" * len(header))
for provider, model, prompt_name, scope, path in runs:
    records, labelled = load_run(path)
    if labelled == 0:
        continue
    per_class = score(records, scope_indices(scope, dataset))["per_class"]
    line = f"{model:<38}{prompt_name:<15}{scope:<11}"
    for label_name in ("Positive", "Negative", "Neutral", "Mixed"):
        width = {"Positive": 10, "Negative": 10, "Neutral": 9, "Mixed": 8}[label_name]
        line += f"{per_class[label_name]['f1']:>{width}.3f}"
    print(line)


## Save combined results

In [ ]:
payload = {}
for (provider, model, prompt_name, scope), entry in summary.items():
    key = f"{provider} | {model} | {prompt_name} | {scope}"
    payload[key] = {
        "provider": provider,
        "model": model,
        "prompt": prompt_name,
        "scope": scope,
        "rows": entry["rows"],
        "rows_labelled": entry["labelled"],
        "predictions_file": entry["path"],
        "metrics": entry["metrics"],
    }
payload["published_reference"] = {
    name: PUBLISHED_BASELINES[name]["test"] for name in ("bert", "mbert", "svm")
}

OUT_PATH = OUTPUT_DIR / "prompting_results_table.json"
OUT_PATH.write_text(json.dumps(payload, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
print("saved:", OUT_PATH)
print("runs included:", len(summary))
